In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [3]:
# 1. Load your dataset
df = pd.read_csv('upi_fraud_dataset_2000_v2.csv')

# 2. Prepare features and target
X = df.drop(columns=['transaction_id', 'is_fraud'])
X = pd.get_dummies(X, drop_first=True)
y = df['is_fraud']

In [4]:
# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# 5. Build the simple ANN model (Without Dropout)
model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
# 6. Train the model
model.fit(X_train_scaled, y_train, epochs=30, batch_size=32, verbose=1)



Epoch 1/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6106 - loss: 0.6870
Epoch 2/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7038 - loss: 0.5962
Epoch 3/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7237 - loss: 0.5280
Epoch 4/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7688 - loss: 0.4647
Epoch 5/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8138 - loss: 0.4120
Epoch 6/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8331 - loss: 0.3794
Epoch 7/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8413 - loss: 0.3608
Epoch 8/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8462 - loss: 0.3517
Epoch 9/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8444 - loss: 0.3453
Epoch 10/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8462 - loss: 0.3413
Epoch 11/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8481 - loss: 0.3379
Epoch 12/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8487 - lo

In [7]:
# 7. Evaluate accuracy
y_pred = (model.predict(X_test_scaled) > 0.5).astype(int).flatten()
accuracy = accuracy_score(y_test, y_pred)
print(f"\nANN Test Accuracy: {accuracy * 100:.2f}%")

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 

ANN Test Accuracy: 83.25%


In [8]:
# Define the new transaction using original categorical features
new_transaction_raw = pd.DataFrame([{
    'amount': 3000.0,
    'transaction_type': 'UPI',  # original categorical feature
    'hour': 2,
    'location_match': 0,
    'device_match': 0,
    'daily_transactions': 12,
    'failed_attempts': 3,
    'new_beneficiary': 1
}])

# Match the dummy encoding of the training set
new_transaction = pd.get_dummies(new_transaction_raw)
new_transaction = new_transaction.reindex(columns=X.columns, fill_value=0)


In [12]:
# Scale the new input
new_transaction_scaled = scaler.transform(new_transaction)

# Get prediction probability
fraud_prob = model.predict(new_transaction_scaled)[0][0]
print(fraud_prob)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
0.9987213


In [10]:
# Using an if statement to classify the transaction
if fraud_prob > 0.5:
    print("Result: FRAUD TRANSACTION 🚨")
else:
    print("Result: NORMAL TRANSACTION ✅")

Result: FRAUD TRANSACTION 🚨
